In [1]:
!pip install google.generativeai
import os
from dotenv import load_dotenv
import json
import gradio as gr
import google.generativeai as genai


[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
load_dotenv(override = True)
api_key = os.getenv("GEM_API_KEY")
genai.configure(api_key=api_key)

In [3]:
model = genai.GenerativeModel("gemini-2.5-flash")

In [4]:
system_message = "You are a very helpful Airline Assistant called FlightAI"

In [5]:
def chat(message,history):
    messages = [{"role":"system", "content":system_message}]
    for human, assistant in history:
        messages.append({"role":"user", "content":human})
        messages.append({"role":"assistant","content":assistant})
    messages.append({"role":"user","content":message})
    response = model.generate_content(system_message)
    return response.text

gr.ChatInterface(fn=chat).launch()

C:\Users\hp\anaconda3\Lib\site-packages\gradio\chat_interface.py:345: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  self.chatbot = Chatbot(


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


In [6]:
ticket_price = {"london":"$799", "paris":"$899", "tokyo":"$1400","usa":"$999"}

def get_ticket(destination_city):
    city = destination_city.lower()
    return ticket_price.get(city,"unknown")

In [7]:
get_ticket("london")

'$799'

In [8]:
"""
Dictionary specification describing the `get_ticket_price` function for use in function calling APIs.

This structure follows the JSON Schema format and is typically used with AI assistants 
(e.g., OpenAI, Gemini) to allow them to programmatically call a predefined function 
when a specific intent is detected, such as asking for ticket prices.

Attributes:
    name (str): The function's identifier ("get_ticket_price").
    description (str): Explains the purpose of the function and when it should be used.
    parameters (dict): JSON Schema defining the expected input parameters:
        type (str): Always "object" to indicate the function takes a JSON object as input.
        properties (dict):
            destination_city (dict):
                type (str): The city the user wants to travel to.
                description (str): Explains that this is the destination for ticket pricing.
        required (list): Specifies that "destination_city" must be provided.
        additionalProperties (bool): Disallows parameters other than those defined.
"""
price_function = {
    "name": "get_ticket_price",
    "description": "Get the price of a return ticket to the destination city. Call this whenever you need to know the ticket price, for example when a customer asks 'How much is a ticket to this city'",
    "parameters": {
        "type": "object",
        "properties": {
            "destination_city": {
                "type": "string",
                "description": "The city that the customer wants to travel to",
            },
        },
        "required": ["destination_city"],
        "additionalProperties": False
    }
}

In [9]:
tools = [{"type": "function", "function": price_function}]

In [10]:
def chat(message, history):
    """
    Handles a single turn of chat with the Gemini API, including tool call handling.

    Args:
        message (str): The latest user message.
        history (list): Conversation history in Gemini-compatible role/parts format.

    Returns:
        str: Model's final reply text.
    """

    messages = [{"role": "user", "parts": [system_message]}]

    
    for msg in history:
        if msg["role"] == "user":
            messages.append({"role": "user", "parts": [msg["content"]]})
        elif msg["role"] == "assistant":
            messages.append({"role": "model", "parts": [msg["content"]]})


    messages.append({"role": "user", "parts": [message]})
    response = model.generate_content(messages)

    if response.candidates and "functionCall" in str(response):
        # Example: Detecting a function call from the response
        tool_response, city = handle_tool_call(response)
        messages.append({"role": "model", "parts": [str(response.text)]})
        messages.append({"role": "user", "parts": [tool_response]})
        response = model.generate_content(messages)

    return (response.text or "").strip()


In [11]:
def handle_tool_call(response):
    """
    Handles a Gemini function/tool call by parsing the model's structured output
    and returning the tool's execution result.

    Args:
        response: Gemini model's response object from `generate_content`.

    Returns:
        tuple: (tool_response_dict, city)
            tool_response_dict -> dict containing the tool's reply for the conversation
            city -> str, extracted destination city
    """
    parts = response.candidates[0].content.parts

    function_call = None
    for part in parts:
        if "functionCall" in part:
            function_call = part["functionCall"]
            break

    if not function_call:
        raise ValueError("No function call found in Gemini response")

    args = function_call.get("args", {})
    city = args.get("destination_city")

    price = get_ticket_price(city)

    tool_response = {
        "role": "function",
        "parts": [{
            "functionResponse": {
                "name": function_call["name"],
                "response": {"destination_city": city, "price": price}
            }
        }]
    }
    return tool_response, city


In [12]:
gr.ChatInterface(fn=chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


In [25]:
!pip install diffusers transformers accelerate torch torchvision pillow

import torch
from diffusers import DiffusionPipeline
from PIL import Image
import requests
from io import BytesIO


[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [26]:
LIGHT_MODEL = "segmind/tiny-sd"  
device = "cuda" if torch.cuda.is_available() else "cpu"

def initialize_light_pipeline():
    """Initialize a lightweight Stable Diffusion pipeline."""
    try:
        pipe = DiffusionPipeline.from_pretrained(
            LIGHT_MODEL,
            torch_dtype=torch.float16 if device == "cuda" else torch.float32,
            safety_checker=None,
            requires_safety_checker=False,
            use_safetensors=True
        )
        pipe = pipe.to(device)
        
        if device == "cuda":
            pipe.enable_attention_slicing()  
            pipe.enable_model_cpu_offload()  
        
        return pipe
    except Exception as e:
        print(f"Error initializing light pipeline: {e}")
        return None

def artist_api(city: str, hf_token: str = None) -> Image.Image:
    """
    Fast image generation using Hugging Face Inference API.
    
    Args:
        city (str): Name of the destination city
        hf_token (str): Your Hugging Face token (get from https://huggingface.co/settings/tokens)
    
    Returns:
        PIL.Image.Image: Generated image
    """
    
    if not hf_token:
        API_URL = "https://api-inference.huggingface.co/models/runwayml/stable-diffusion-v1-5"
        headers = {}
    else:
        API_URL = "https://api-inference.huggingface.co/models/runwayml/stable-diffusion-v1-5"
        headers = {"Authorization": f"Bearer {hf_token}"}
    
    prompt = f"pop art vacation poster of {city}, colorful, iconic landmarks, travel poster style"
    
    payload = {"inputs": prompt}
    
    try:
        response = requests.post(API_URL, headers=headers, json=payload, timeout=60)
        response.raise_for_status()
        
        image = Image.open(BytesIO(response.content))
        return image
        
    except Exception as e:
        raise ValueError(f"API Error: {str(e)}")


light_pipeline = None

def artist_fast(city: str) -> Image.Image:
    """
    Fast local image generation with minimal quality settings.
    
    Args:
        city (str): Name of the destination city
    
    Returns:
        PIL.Image.Image: Generated image (lower quality but fast)
    """
    global light_pipeline
    
    if light_pipeline is None:
        light_pipeline = initialize_light_pipeline()
    
    if light_pipeline is None:
        raise ValueError("Pipeline initialization failed")
    
    prompt = f"{city} vacation poster, pop art style, colorful"
    
    try:
        image = light_pipeline(
            prompt=prompt,
            num_inference_steps=10,   
            guidance_scale=5.0,        
            width=256,                 
            height=256,
            num_images_per_prompt=1
        ).images[0]
        
        image = image.resize((512, 512), Image.Resampling.LANCZOS)
        return image
        
    except Exception as e:
        raise ValueError(f"Generation error: {str(e)}")


def artist_replicate(city: str) -> Image.Image:
    """
    Ultra-fast generation using Replicate API (requires account).
    
    Args:
        city (str): Name of the destination city
    
    Returns:
        PIL.Image.Image: Generated image
    """
    try:
        import replicate
        
        output = replicate.run(
            "stability-ai/stable-diffusion:27b93a2413e7f36cd83da926f3656280b2931564ff050bf9575f1fdf9bcd7478",
            input={
                "prompt": f"pop art vacation poster of {city}, colorful travel poster",
                "num_inference_steps": 20,
                "width": 512,
                "height": 512
            }
        )
        
        # Download the image
        response = requests.get(output[0])
        image = Image.open(BytesIO(response.content))
        return image
        
    except ImportError:
        raise ValueError("Install replicate: pip install replicate")
    except Exception as e:
        raise ValueError(f"Replicate error: {str(e)}")

def artist_simple(city: str) -> Image.Image:
    """
    Simplest possible implementation using public APIs.
    
    Args:
        city (str): Name of the destination city
    
    Returns:
        PIL.Image.Image: Generated image
    """
    try:
        prompt = f"pop art vacation poster of {city} colorful travel".replace(" ", "%20")
        url = f"https://image.pollinations.ai/prompt/{prompt}?width=512&height=512"
        
        response = requests.get(url, timeout=30)
        response.raise_for_status()
        
        image = Image.open(BytesIO(response.content))
        return image
        
    except Exception as e:
        raise ValueError(f"Simple generation error: {str(e)}")


def speed_test():
    """Test different methods and compare speeds."""
    import time
    
    city = "Paris"
    methods = [
        ("Simple API", artist_simple),
        ("HF API", lambda c: artist_api(c, hf_token=None)),
        # ("Fast Local", artist_fast),  # Uncomment if you have GPU
    ]
    
    for name, method in methods:
        try:
            print(f"\n Testing {name}...")
            start_time = time.time()
            
            image = method(city)
            
            end_time = time.time()
            duration = end_time - start_time
            
            print(f"{name}: {duration:.2f} seconds")
            image.save(f"{name.lower().replace(' ', '_')}_{city}.png")
            
        except Exception as e:
            print(f"{name} failed: {e}")

def artist(city: str, method: str = "auto") -> Image.Image:
    """
    Main artist function with multiple speed options.
    
    Args:
        city (str): Name of the destination city
        method (str): Generation method - "auto", "simple", "api", "fast", "replicate"
    
    Returns:
        PIL.Image.Image: Generated image
    """
    methods = {
        "simple": artist_simple,
        "api": lambda c: artist_api(c, hf_token=None),
        "fast": artist_fast,
        "replicate": artist_replicate
    }
    
    if method == "auto":
        for method_name in ["simple", "api", "fast"]:
            try:
                print(f"Trying {method_name} method...")
                return methods[method_name](city)
            except Exception as e:
                print(f"{method_name} failed: {e}")
                continue
        raise ValueError("All methods failed")
    
    elif method in methods:
        return methods[method](city)
    
    else:
        raise ValueError(f"Unknown method: {method}. Use: {list(methods.keys())}")

if __name__ == "__main__":
    try:
        print("Generating image for india...")
        image = artist("india", method="simple")
        image.save("paris.png")
        image.show()
        print("Done!")
        
    except Exception as e:
        print(f"Error: {e}")

Generating image for india...
Done!
